# M6: Validation and sensitivity checks

This notebook implements the four validation strands described in M6 of the
paper:

**(a) Human inter-rater validation** — a stratified random sample of
paragraphs coded by a human alongside the LLM, to assess reliability of the
concern extraction step.

**(b) Prompt sensitivity** — three prompt variants (V0/V1/V3) applied to a
stratified sample of chunks, to test whether the headline findings are
artefacts of specific prompt wording.

**(c) Lens-scheme dendrogram validation** — computed in
`02_shared_structure.ipynb`; results (ARI = 0.185 concerns, 0.150 benefits)
are reported in M6(c) and in R4 Limitations.

**(d) k-sensitivity** — headline outputs re-run at k=60 and k=90 (baseline
k=75), to test whether substantive conclusions depend on clustering
resolution.

This notebook also exports traceability datasets and the evidence pack.
Artifacts from `01_processing.ipynb` and `01a_clustering.ipynb` are loaded;
no API calls are needed except for the prompt sensitivity cell (§SECTION 10C).

In [ ]:
# @title Load pre-computed artifacts
# Run this cell before any analysis cell.  It loads all outputs written by
# 01_processing.ipynb from disk so this notebook never calls the OpenAI API
# or re-runs k-means.
from pathlib import Path
import pub_dialogue.utils as du
from pub_dialogue.utils import show_status, show_complete, AccessStage, AddressStage
import pandas as pd
import numpy as np

# --- CIP-0010: stage objects centralise all path / parameter constants ---
_access  = AccessStage()
_address = AddressStage(access=_access)
OUTPUT_FOLDER     = _access.output_folder
CHECKPOINT_FOLDER = _access.checkpoint_folder
TECH_COL          = _address.tech_col

a = _access.load_artifacts()

chunks_df    = a["chunks_df"]
concerns_df  = a["concerns_df"]
benefits_df  = a["benefits_df"]

concern_embeddings  = a["concern_embeddings"]
benefit_embeddings  = a["benefit_embeddings"]
concern_centroids   = a["concern_centroids"]
benefit_centroids   = a["benefit_centroids"]

concern_ids          = a["concern_ids"]
benefit_ids          = a["benefit_ids"]
cluster_labels       = a["cluster_labels"]
benefit_cluster_labels = a["benefit_cluster_labels"]
cluster_summary_df   = a["cluster_summary_df"]
benefit_cluster_summary_df = a["benefit_cluster_summary_df"]

framing_lens_mappings         = a["framing_lens_mappings"]
benefit_framing_lens_mappings = a["benefit_framing_lens_mappings"]

cluster_entropy           = a["cluster_entropy"]
cluster_entropy_norm      = a["cluster_entropy_norm"]
cross_cutting_clusters    = a["cross_cutting_clusters"]

benefit_cluster_entropy          = a["benefit_cluster_entropy"]
normalized_entropy_benefits      = a["normalized_entropy_benefits"]
cross_cutting_clusters_benefits  = a["cross_cutting_clusters_benefits"]

# Convenience: CLUSTER_LABELS / BENEFIT_CLUSTER_LABELS dicts used by plots
CLUSTER_LABELS        = {int(k): v for k, v in cluster_labels.items()}
BENEFIT_CLUSTER_LABELS = {int(k): v for k, v in benefit_cluster_labels.items()}

# Re-apply technology_meta merge if not already present in the loaded CSV
if "technology_meta" not in concerns_df.columns:
    import pandas as pd
    _tech = chunks_df[["chunk_id", "technology_meta"]]
    concerns_df = concerns_df.merge(_tech, on="chunk_id", how="left")
    benefits_df = benefits_df.merge(_tech, on="chunk_id", how="left")

print(f"Artifacts loaded from {OUTPUT_FOLDER} / {CHECKPOINT_FOLDER}")
print(f"  Chunks: {len(chunks_df):,}  |  Concerns: {len(concerns_df):,}  |  Benefits: {len(benefits_df):,}")

## Import analysis helpers

In [ ]:
# clusters_to_labels, clusters_to_lenses, html_escape — imported from dialogue_utils (see Cell 5)
pass

## M6(a): Human inter-rater validation

**Design**: a stratified random sample of paragraphs, drawn to ensure
representation across technologies, is coded by a human coder using a simple
binary decision: "does this paragraph express a concern?"

**Results from v19 run**:
- Sample size: 50 paragraphs (proof of concept; expanding to a larger sample
  is an open item before submission)
- Raw agreement: 92%
- Cohen's κ = 0.84 (almost-perfect agreement on the Landis–Koch scale)
- **Asymmetric error**: LLM over-extracts (4 cases where LLM found a concern
  but the human did not) and does not under-extract (0 cases where the human
  found a concern the LLM missed). The bias has a direction: when uncertain,
  the LLM tends to find concerns rather than miss them.
- Procedural and methodological text is occasionally mined for concerns it
  doesn't substantively contain.

**Implication**: the extraction is reliable; over-extraction is a minor and
directionally predictable bias. The overall concern-phrase counts should be
treated as slight over-estimates.

In [ ]:
# @title (v16) Export human validation sample (concerns)
# Issue 12 from the audit: the paper makes methodological claims about LLM-
# mediated qualitative scaling but does not include a human inter-rater check.
# The traceability output above gives every paragraph + extracted concerns +
# cluster + lens. This cell exports a stratified-by-technology random sample
# of paragraphs ready for hand-coding.
#
# To use: download the Excel file, fill the human columns, re-import, and
# compute agreement statistics (Cohen's kappa or simple agreement) against
# the LLM extractions.

show_status("Exporting human validation sample (concerns)...")

VALIDATION_SAMPLE_N = 250
RANDOM_SEED = _address.random_seed  # CIP-0010: derived from stage object

# Stratified sample by technology, with a floor per technology so small-N
# technologies are not invisible.
per_tech_floor = 10
sample_pieces = []
for tech, group in chunks_df.groupby("technology_meta"):
    n_for_tech = max(per_tech_floor, int(round(VALIDATION_SAMPLE_N * len(group) / len(chunks_df))))
    n_for_tech = min(n_for_tech, len(group))
    sample_pieces.append(group.sample(n=n_for_tech, random_state=RANDOM_SEED))

validation_sample = pd.concat(sample_pieces, ignore_index=True)
# Trim if oversized after the floor
if len(validation_sample) > VALIDATION_SAMPLE_N * 1.5:
    validation_sample = validation_sample.sample(int(VALIDATION_SAMPLE_N * 1.5), random_state=RANDOM_SEED)

# Attach LLM extractions for each sampled chunk
sample_concerns = (concerns_df.groupby("chunk_id")["concern"]
                   .apply(lambda s: " | ".join(s))
                   .rename("llm_concerns"))
validation_sample = validation_sample.merge(sample_concerns, on="chunk_id", how="left")
validation_sample["llm_concerns"] = validation_sample["llm_concerns"].fillna("(NO_CONCERN)")
validation_sample["llm_concern_count"] = validation_sample["llm_concerns"].apply(
    lambda s: 0 if s == "(NO_CONCERN)" else s.count("|") + 1
)

# Empty columns for the human coder
for col in ["human_any_concern", "human_concerns", "human_retain_llm_extraction", "human_notes"]:
    validation_sample[col] = ""

cols_for_export = [
    "chunk_id", "source_file", "technology_meta", "year", "word_count",
    "text", "llm_concerns", "llm_concern_count",
    "human_any_concern", "human_concerns", "human_retain_llm_extraction", "human_notes",
]
validation_sample = validation_sample[cols_for_export]

from pub_dialogue.address import _clean_for_xlsx

validation_sample_xlsx = validation_sample.map(_clean_for_xlsx)
validation_sample_xlsx.to_excel(OUTPUT_FOLDER / "human_validation_sample_concerns.xlsx", index=False)
validation_sample.to_csv(OUTPUT_FOLDER / "human_validation_sample_concerns.csv", index=False)

print(f"Exported {len(validation_sample)} chunks for human validation.")
print(f"Stratified by technology; per-technology floor = {per_tech_floor}.")
print()
print("Suggested human columns:")
print("  human_any_concern: yes / no / unclear")
print("  human_concerns: pipe-separated phrases the human extracted")
print("  human_retain_llm_extraction: yes / no / partial")
print("  human_notes: free text")

show_complete("Human validation sample exported. See human_validation_sample_concerns.xlsx")

In [ ]:
# @title Export traceability datasets and evidence pack (concerns)
show_status('Exporting concern paragraph-level traceability datasets (evidence pack)...')

concern_exports = _address.export_evidence_pack(
    chunks_df=chunks_df,
    phrases_df=concerns_df,
    kind='concern',
    cluster_labels=CLUSTER_LABELS,
    framing_lens_mappings=framing_lens_mappings,
    output_folder=OUTPUT_FOLDER,
    tech_col=TECH_COL,
)

for fmt, path in concern_exports.items():
    print(f'  Written ({fmt}): {path}')
show_complete('Concern traceability exports written (CSV/JSONL/Parquet + HTML evidence pack)')

## Import entropy and fingerprint helpers

In [ ]:
# @title Export traceability datasets and evidence pack (benefits)
show_status('Exporting benefit paragraph-level traceability datasets (evidence pack)...')

benefit_exports = _address.export_evidence_pack(
    chunks_df=chunks_df,
    phrases_df=benefits_df,
    kind='benefit',
    cluster_labels=BENEFIT_CLUSTER_LABELS,
    framing_lens_mappings=benefit_framing_lens_mappings,
    output_folder=OUTPUT_FOLDER,
    tech_col=TECH_COL,
)

for fmt, path in benefit_exports.items():
    print(f'  Written ({fmt}): {path}')
show_complete('Benefit traceability exports written (CSV/JSONL/Parquet + HTML evidence pack)')

In [ ]:
# normalized_entropy, ai_fingerprint_over_crosscut — imported from dialogue_utils (see Cell 5)
pass

## Concern ranking volatility: mean absolute rank change

Measures how stable the ranking of concern clusters is year-to-year. A low
mean absolute rank change indicates that the same clusters are consistently
prominent, supporting the "stable core" interpretation. High volatility for
specific clusters indicates their prominence is sensitive to which documents
happen to fall in a given year — a further reason to prefer the
four-window analysis over year-by-year claims.

In [ ]:
# @title Compute AI concern year × cluster matrix (ai_year)
# ai_year: rows = years, columns = concern cluster IDs,
# values = FRACTION OF DOCUMENTS in that year mentioning that cluster.
# Methodology: CIP-0009 Approach B (document-level binary weighting).
# CIP-0010: delegated to _address.concern_year_matrix()

ai_year = _address.concern_year_matrix(concerns_df, chunks_df)
years = sorted(ai_year.index.tolist())

print(f"ai_year: {ai_year.shape}  (years × concern clusters for AI, fraction of docs)")


## Concern ranking volatility: visualisation

In [ ]:
# @title Assess volatility of AI concern rankings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


ai_rel = ai_year.div(ai_year.sum(axis=1), axis=0).fillna(0)

ranks = ai_rel.rank(axis=1, ascending=False, method="average")

rank_diff = ranks.diff().abs()

volatility = rank_diff.mean(axis=1)

vol_df = pd.DataFrame({
    "year": volatility.index,
    "mean_rank_change": volatility.values
}).dropna()

plt.figure(figsize=(9, 4.5))
plt.plot(vol_df["year"], vol_df["mean_rank_change"], marker="o")

plt.xlabel("Year")
plt.ylabel("Mean absolute rank change")
plt.title("Stability of AI public concerns over time\n(lower values = greater normalisation)")
plt.tight_layout()
plt.show()

display(vol_df)


In [ ]:
# @title Assess stable-core robustness (concerns)
show_status('Assessing stable-core robustness of AI concern fingerprint...')

salience_df = _address.concern_salience(concerns_df)
robustness_summary, ai_fingerprint = _address.stable_core_robustness(salience_df)

display(robustness_summary)
show_complete('Stable-core concern robustness complete')

## Import year-parsing and tokenisation helpers

In [ ]:
# parse_year, tokenize — imported from dialogue_utils (see Cell 5)
pass

## Novelty of concern types over time

How much of the concern cluster space is "new" each year (clusters not
previously active)? This is a supplementary analysis that checks whether
the shifting-content finding in R3 reflects genuine emergence of new concern
types or just changing prominence of clusters that were always present.

A low novelty rate would support the interpretation that the stable-core
finding is robust: the same concerns recur throughout the period, with
changing emphasis rather than wholesale replacement of concern types.

In [ ]:
# @title Assess novelty of concern types in AI dialogues

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ai_year exists: years × clusters (raw salience)
present = (ai_year > 0).astype(int)

seen = set()
rows = []
for y in present.index:
    active = set(present.columns[present.loc[y] > 0])
    new = active - seen
    rows.append({
        "year": int(y),
        "active_clusters": len(active),
        "new_clusters": len(new),
        "new_cluster_share": len(new) / max(len(active), 1)
    })
    seen |= active

nov2 = pd.DataFrame(rows)

plt.figure(figsize=(9, 4.5))
plt.plot(nov2["year"], nov2["new_cluster_share"], marker="o")
plt.xlabel("Year")
plt.ylabel("Share of newly appearing concern clusters")
plt.title("Novelty of concern-types in AI discourse over time")
plt.tight_layout()
plt.show()

display(nov2)


In [ ]:
# @title Lexical novelty of concern phrases over time (AI)
show_status('Computing lexical novelty of AI concern phrases over time...')

novelty_concern_df = _address.lexical_novelty_over_time(concerns_df, 'concern')

display(novelty_concern_df)
show_complete('Lexical novelty of concern phrases complete')

---
# SECTION 10: M6(d) Sensitivity analysis: number of clusters (concerns)

We re-run *selected headline outputs* for **k = 60** and **k = 90** (baseline
**k = 75**) to assess whether substantive conclusions depend on clustering
resolution.

**Result (M6d)**: headline findings are preserved across all three k values.
The qualitative pattern of AI distinctiveness — over-indexed on
Privacy/Consent/Data, under-indexed on Safety/Health/Environment — holds at
k=60, k=75, and k=90. Cluster-level pp differences shift slightly in
magnitude but the rank order is stable.

**Notes on method:**
- Cluster IDs are not expected to match across k.
- For framing-lens comparisons, we map new clusters to the closest baseline
  (k=75) cluster centroid (cosine similarity) and inherit baseline lens
  memberships. This provides a consistent lens vocabulary without re-running
  LLM labelling.


In [ ]:
# clusters_to_labels, clusters_to_lenses, html_escape — imported from dialogue_utils (see Cell 5)
pass

## Data cleanliness: strip control characters before export

Removes illegal control characters from PDF extraction artifacts before
writing to Excel. Required before any export step in the k-sensitivity
analysis to prevent corrupt xlsx output.

In [ ]:
# _clean_for_xlsx is defined locally in the export cell above (Cell 94).
# It strips openpyxl-illegal control characters (PDF extraction artefacts)
# from Excel export values and is NOT imported from dialogue_utils.

## Import entropy and fingerprint helpers (k-sensitivity)

In [ ]:
# normalized_entropy, ai_fingerprint_over_crosscut — imported from dialogue_utils (see Cell 5)
pass

## Benefit ranking volatility: mean absolute rank change

In [ ]:
# @title Compute AI benefit year × cluster matrix (benefit_ai_year)
# benefit_ai_year: rows = years, columns = benefit cluster IDs,
# values = FRACTION OF DOCUMENTS in that year mentioning that cluster.
# Same document-level binary weighting as ai_year (CIP-0009 Approach B).
# CIP-0010: delegated to _address.benefit_year_matrix()

benefit_ai_year = _address.benefit_year_matrix(benefits_df, chunks_df)
benefit_years = sorted(benefit_ai_year.index.tolist())

print(f"benefit_ai_year: {benefit_ai_year.shape}  (years × benefit clusters for AI, fraction of docs)")


## Benefit ranking volatility: visualisation

In [ ]:
# @title Assess volatility of AI benefit rankings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "benefit_ai_year" not in globals():
    raise NameError("benefit_ai_year not found. Run cell 80 first.")

ai_rel = benefit_ai_year.div(benefit_ai_year.sum(axis=1), axis=0).fillna(0)

ranks = ai_rel.rank(axis=1, ascending=False, method="average")

rank_diff = ranks.diff().abs()

volatility = rank_diff.mean(axis=1)

vol_df = pd.DataFrame({
    "year": volatility.index,
    "mean_rank_change": volatility.values
}).dropna()

plt.figure(figsize=(9, 4.5))
plt.plot(vol_df["year"], vol_df["mean_rank_change"], marker="o")

plt.xlabel("Year")
plt.ylabel("Mean absolute rank change")
plt.title("Stability of AI public benefits over time\n(lower values = greater normalisation)")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_ai_volatility_over_time.png", dpi=150, bbox_inches='tight')
plt.show()

display(vol_df)


In [ ]:
# @title Assess stable-core robustness (benefits)
show_status('Assessing stable-core robustness of AI benefit fingerprint...')

benefit_salience_df = _address.benefit_salience(benefits_df)
benefit_robustness_summary, benefit_ai_fingerprint = _address.stable_core_robustness(benefit_salience_df)

display(benefit_robustness_summary)
show_complete('Stable-core benefit robustness complete')

## Import year-parsing and tokenisation helpers (benefits)

In [ ]:
# parse_year, tokenize — imported from dialogue_utils (see Cell 5)
pass

## Novelty of benefit types over time

In [ ]:
# @title Assess novelty of benefit types in AI dialogues

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

present = (benefit_ai_year > 0).astype(int)

seen = set()
rows = []
for y in present.index:
    active = set(present.columns[present.loc[y] > 0])
    new = active - seen
    rows.append({
        "year": int(y),
        "active_clusters": len(active),
        "new_clusters": len(new),
        "new_cluster_share": len(new) / max(len(active), 1)
    })
    seen |= active

nov2 = pd.DataFrame(rows)

plt.figure(figsize=(9, 4.5))
plt.plot(nov2["year"], nov2["new_cluster_share"], marker="o")
plt.xlabel("Year")
plt.ylabel("Share of newly appearing benefit clusters")
plt.title("Novelty of benefit-types in AI discourse over time")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_ai_cluster_novelty.png", dpi=150, bbox_inches='tight')
plt.show()

display(nov2)


In [ ]:
# @title Lexical novelty of benefit phrases over time (AI)
show_status('Computing lexical novelty of AI benefit phrases over time...')

novelty_benefit_df = _address.lexical_novelty_over_time(benefits_df, 'benefit')

display(novelty_benefit_df)
show_complete('Lexical novelty of benefit phrases complete')

---
# SECTION 10B: M6(d) Sensitivity analysis: number of clusters (benefits)

Benefit-side equivalent of §SECTION 10. Re-runs selected headline outputs at
k=60 and k=90 to confirm that the benefit mirror finding (AI over-indexed on
Service Efficiency & Personalisation, under-indexed on Safety/Risk/Environment)
is not an artefact of the k=75 choice.

**Notes:**
- Cluster IDs are not expected to match across k.
- For framing-lens comparisons, we map new clusters to the closest baseline
  (k=75) cluster centroid (cosine similarity) and inherit baseline lens
  memberships. This provides a consistent lens vocabulary without re-running
  LLM labelling.


---
# SECTION 10C: M6(b) Sensitivity analysis: extraction prompt wording

We test 3 prompt variants per kind (concern, benefit) on a stratified sample
of 529 paragraphs to assess whether headline findings are artefacts of
specific prompt wording.

**Variants tested** (concern side; benefit side mirrors):
- **V0 (current / baseline):** The production prompt — structured rules,
  negative examples, decontextualisation instruction, NO_CONCERN sentinel.
- **V1 (broader framing):** "Issues" framing instead of "concerns";
  semantically broader to capture more ambivalent expressions.
- **V3 (strict decontextualisation):** Stricter instruction to remove
  technology-specific language; designed to test the upper bound on
  decontextualisation.

**Key result (M6b, run against v16 corpus)**:
- The qualitative pattern of AI distinctiveness is preserved across all three
  variants. The rank order of over- and under-indexed lenses is the same.
- The **Privacy effect magnitude is sensitive to decontextualisation strictness**:
  V0: +21pp; V1: +16pp; V3: +9pp.
- V0 is an upper bound; V3 is a lower bound on the precise pp difference.
- *Note: this analysis was run against the v16 corpus. Re-running against v19
  chunks is a candidate task before submission; the qualitative direction is
  expected to hold.*

**Thresholds for agreement**:
- **yield_agreement** ≥ 85%: fraction of chunks where both variants agree on
  whether ≥1 phrase was extracted.
- **phrase_agreement** ≥ 70%: among chunks where both variants extracted ≥1
  phrase, the fraction of phrases with a semantic near-match (cosine
  similarity ≥0.85) in the other variant, averaged symmetrically.

If results fall below these thresholds, paper claims should be qualified as
prompt-dependent.

In [ ]:
# @title Prompt sensitivity analysis — concerns
# Requires: paragraph_chunks.csv (from load_artifacts), a configured LLM client
# NOTE: this cell makes LLM API calls (~200 chunks × 3 variants = ~600 calls).
# Cost estimate at gpt-4o-mini: < $0.10 USD.

from pub_dialogue.client import LLMClient
import pub_dialogue.utils as du
from pathlib import Path

OUTPUT_FOLDER = Path("outputs")

# Configure client — reads OPENAI_API_KEY from environment / .env
client = LLMClient(model="gpt-4o-mini")

show_status("Running prompt sensitivity analysis (concerns)…")

concern_sensitivity = du.run_prompt_sensitivity(
    chunks=a["paragraph_chunks"],
    kind="concern",
    client=client,
    sample_n=200,
    random_seed=42,
    output_folder=OUTPUT_FOLDER,
)

print("\nConcern prompt sensitivity results:")
print(concern_sensitivity.to_string(index=False))
show_complete("Prompt sensitivity (concerns) complete")

In [ ]:
# @title Prompt sensitivity analysis — benefits
show_status("Running prompt sensitivity analysis (benefits)…")

benefit_sensitivity = du.run_prompt_sensitivity(
    chunks=a["paragraph_chunks"],
    kind="benefit",
    client=client,
    sample_n=200,
    random_seed=42,
    output_folder=OUTPUT_FOLDER,
)

print("\nBenefit prompt sensitivity results:")
print(benefit_sensitivity.to_string(index=False))
show_complete("Prompt sensitivity (benefits) complete")

In [ ]:
# @title Export all analysis outputs

import os
import zipfile
from pathlib import Path
try:
    from google.colab import files as _colab_files
    _in_colab = True
except ImportError:
    _in_colab = False

# Where outputs live
OUTPUT_DIR = Path(OUTPUT_FOLDER)
ARTIFACT_DIR = Path(globals().get("ARTIFACT_DIR", "analysis_artifacts"))

ZIP_NAME = "public_dialogue_analysis_outputs.zip"
ZIP_PATH = OUTPUT_DIR / ZIP_NAME

print("Preparing ZIP archive...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    # Add OUTPUT_FOLDER contents
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file() and path.name != ZIP_NAME:
            zf.write(path, arcname=f"outputs/{path.relative_to(OUTPUT_DIR)}")

    # Optionally add analysis_artifacts if it exists
    if ARTIFACT_DIR.exists():
        for path in ARTIFACT_DIR.rglob("*"):
            if path.is_file():
                zf.write(path, arcname=f"analysis_artifacts/{path.relative_to(ARTIFACT_DIR)}")

print(f"ZIP written to: {ZIP_PATH}")
print("Files included:")

for p in sorted(OUTPUT_DIR.glob("*")):
    if p.name != ZIP_NAME:
        print(" - outputs/", p.name)

if ARTIFACT_DIR.exists():
    print(" - analysis_artifacts/ (included)")

# Trigger download in Colab
if _in_colab:
    _colab_files.download(str(ZIP_PATH))
else:
    print(f"(Colab download skipped — running locally. ZIP is at {ZIP_PATH})")
